<a href="https://colab.research.google.com/github/eishaiqbal767-cmyk/AI-TMBD-MOVIES-PROJECT/blob/main/Notebooks/TMBD_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE = '/content/drive/MyDrive/AI_Project'
print(os.path.exists(BASE))
print(os.listdir(BASE))

True
['notebooks', 'data', 'reports', 'src']


I mounted my Google Drive in Colab so the notebook can read and save files in my project folder.

In [ ]:
import requests
from google.colab import userdata

TOKEN = userdata.get('TMDB_TOKEN')
BASE_URL = 'https://api.themoviedb.org/3'
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'accept': 'application/json'
}

params = {
    'primary_release_date.gte': '2016-01-01',
    'primary_release_date.lte': '2016-12-31',
    'sort_by': 'vote_count.desc',
    'page': 1
}
r = requests.get(f'{BASE_URL}/discover/movie', headers=HEADERS, params=params)
print('Status code:', r.status_code)
data = r.json()
print('Total results:', data['total_results'])
print('Total pages:', data['total_pages'])
print('Movies on this page:', len(data['results']))
print('First movie:', data['results'][0]['title'], '| id:', data['results'][0]['id'])

Status code: 200
Total results: 20001
Total pages: 1001
Movies on this page: 20
First movie: Deadpool | id: 293660


I made a first test request to the TMDB Discover endpoint to check that my API token works.

# TMDB Movie Data Analysis and Machine Learning

Programming for AI, semester project. Team: [Eisha Iqbal,25L-3003], [Ifrah noor, 25L-0025]. Instructor: Mahhek Tahir.

In this notebook I collect movie data from the TMDB API for the years 2016 to 2026, clean and explore it, and build regression models to predict a numerical target.

In [ ]:
import os, re, json, time, hashlib, random
from datetime import datetime
import numpy as np
import pandas as pd

ROLL_NUMBERS = ['25L-3003', '25L-0025']

SEED = 25

random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/AI_Project'
RAW_DIR = f'{BASE}/data/raw'
PROC_DIR = f'{BASE}/data/processed'
JSON_DIR = f'{RAW_DIR}/json'
os.makedirs(JSON_DIR, exist_ok=True)

START_YEAR, END_YEAR = 2016, 2026
MOVIES_PER_YEAR = 100
SORT_BY = 'vote_count.desc'

print('Roll numbers:', ROLL_NUMBERS)
print('SEED:', SEED)
print('Raw data folder:', RAW_DIR)
print('Years:', START_YEAR, 'to', END_YEAR, '| target per year:', MOVIES_PER_YEAR)
print('Sorting:', SORT_BY)

Roll numbers: ['25L-3003', '25L-0025']
SEED: 25
Raw data folder: /content/drive/MyDrive/AI_Project/data/raw
Years: 2016 to 2026 | target per year: 100
Sorting: vote_count.desc


In [ ]:
COLLECTION_START = datetime.now()
request_count = 0
id_records = []
seen_ids = set()

def api_get(path, params=None):
    global request_count
    for attempt in range(5):
        r = requests.get(f'{BASE_URL}{path}', headers=HEADERS, params=params, timeout=30)
        request_count += 1
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
    raise RuntimeError('Too many failed attempts')

year_summary = []
for year in range(START_YEAR, END_YEAR + 1):
    collected, page, total_pages = 0, 1, None
    while collected < MOVIES_PER_YEAR:
        params = {
            'primary_release_date.gte': f'{year}-01-01',
            'primary_release_date.lte': f'{year}-12-31',
            'sort_by': SORT_BY,
            'vote_count.gte': 10,
            'include_adult': 'false',
            'language': 'en-US',
            'page': page
        }
        data = api_get('/discover/movie', params)
        total_pages = data['total_pages']
        for m in data['results']:
            if m['id'] in seen_ids:
                continue
            seen_ids.add(m['id'])
            id_records.append({'id': m['id'], 'title': m['title'],
                               'discover_year': year, 'discover_page': page})
            collected += 1
            if collected >= MOVIES_PER_YEAR:
                break
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.05)
    year_summary.append({'year': year, 'collected': collected,'pages_used': page - 1
                         if collected >= MOVIES_PER_YEAR else page,
                         'api_total_pages': total_pages})

    print(year, '->', collected, 'movies')

ids_df = pd.DataFrame(id_records)
ids_df.to_csv(f'{RAW_DIR}/movie_ids.csv', index=False)
pd.DataFrame(year_summary).to_csv(f'{RAW_DIR}/discover_summary.csv', index=False)

print('\nUnique IDs:', ids_df['id'].nunique(), '| Requests so far:', request_count)
pd.DataFrame(year_summary)

2016 -> 100 movies
2017 -> 100 movies
2018 -> 100 movies
2019 -> 100 movies
2020 -> 100 movies
2021 -> 100 movies
2022 -> 100 movies
2023 -> 100 movies
2024 -> 100 movies
2025 -> 100 movies
2026 -> 100 movies

Unique IDs: 1100 | Requests so far: 55


,year,collected,pages_used,api_total_pages
0,2016,100,5,173
1,2017,100,5,185
2,2018,100,5,186
3,2019,100,5,184
4,2020,100,5,142
5,2021,100,5,148
6,2022,100,5,160
7,2023,100,5,153
8,2024,100,5,129
9,2025,100,5,102


The Discover step returned 1100 unique movie IDs, exactly 100 for each year from 2016 to 2026, with no duplicates. Each year needed 5 pages of 20 results, and the 11 years together used 55 requests.


In [ ]:
total = len(ids_df)
done_now, skipped = 0, 0

for i, movie_id in enumerate(ids_df['id'], start=1):
    path = f'{JSON_DIR}/{movie_id}.json'
    if os.path.exists(path):
        skipped += 1
        continue
    data = api_get(f'/movie/{movie_id}',
                   {'append_to_response': 'credits', 'language': 'en-US'})
    wrapper = {'collected_at': datetime.now().isoformat(timespec='seconds'),
               'data': data}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(wrapper, f)
    done_now += 1
    if i % 100 == 0:
        print(f'{i}/{total} processed')
    time.sleep(0.05)

COLLECTION_END = datetime.now()
files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
print('\nDownloaded now:', done_now, '| skipped (already saved):', skipped)
print('JSON files on Drive:', len(files))
print('Total requests (discover + details):', request_count)
print('Start:', COLLECTION_START.isoformat(timespec='seconds'))
print('End:  ', COLLECTION_END.isoformat(timespec='seconds'))

with open(f'{RAW_DIR}/collection_log.json', 'w') as f:
    json.dump({'roll_numbers': ROLL_NUMBERS,
               'start': COLLECTION_START.isoformat(timespec='seconds'),
               'end': COLLECTION_END.isoformat(timespec='seconds'),
               'requests': request_count}, f)

100/1100 processed
200/1100 processed
300/1100 processed
400/1100 processed
500/1100 processed
600/1100 processed
700/1100 processed
800/1100 processed
900/1100 processed
1000/1100 processed
1100/1100 processed

Downloaded now: 1100 | skipped (already saved): 0
JSON files on Drive: 1100
Total requests (discover + details): 1155
Start: 2026-11-21T19:51:32
End:   2026-11-21T20:07:55


For each of the 1100 movie IDs I called the Movie Details endpoint and asked for the credits in the same request, so I got budget, revenue, runtime, genres, languages, production companies, cast and crew without extra calls.

In [5]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, re, json, time, hashlib, random, requests
from datetime import datetime
import numpy as np
import pandas as pd

ROLL_NUMBERS = ['25L-3003', '25L-0025']
SEED = 25
random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/AI_Project'
RAW_DIR = f'{BASE}/data/raw'
PROC_DIR = f'{BASE}/data/processed'
JSON_DIR = f'{RAW_DIR}/json'

START_YEAR, END_YEAR = 2016, 2026
MOVIES_PER_YEAR = 100
SORT_BY = 'vote_count.desc'

try:
    TOKEN = userdata.get('TMDB_TOKEN')
except Exception:
    TOKEN = None   # not needed after data collection
BASE_URL = 'https://api.themoviedb.org/3'
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'accept': 'application/json'}

ids_df = pd.read_csv(f'{RAW_DIR}/movie_ids.csv')
RAW_CSV = f'{RAW_DIR}/tmdb_raw.csv'
raw_df = pd.read_csv(RAW_CSV)
print('raw_df reloaded:', raw_df.shape)
print('IDs reloaded:', len(ids_df))
print('JSON files on Drive:', len([f for f in os.listdir(JSON_DIR) if f.endswith('.json')]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
raw_df reloaded: (1100, 28)
IDs reloaded: 1100
JSON files on Drive: 1100


The Colab session restarted, so the variables in memory were lost. I remounted Google Drive, restored my settings, and reloaded the list of movie IDs from the file I had saved earlier.

In [ ]:
def names(items, key='name'):
    if not items:
        return None
    return '|'.join(str(x[key]) for x in items if x.get(key))

rows = []
for movie_id, disc_year in zip(ids_df['id'], ids_df['discover_year']):
    with open(f'{JSON_DIR}/{movie_id}.json', encoding='utf-8') as f:
        w = json.load(f)
    d, credits = w['data'], w['data'].get('credits', {})
    crew, cast = credits.get('crew', []), credits.get('cast', [])
    directors = [c['name'] for c in crew if c.get('job') == 'Director']
    coll = d.get('belongs_to_collection')
    rows.append({
        'id': d['id'],
        'title': d.get('title'),
        'original_title': d.get('original_title'),
        'release_date': d.get('release_date'),
        'discover_year': disc_year,
        'status': d.get('status'),
        'original_language': d.get('original_language'),
        'runtime': d.get('runtime'),
        'budget': d.get('budget'),
        'revenue': d.get('revenue'),
        'popularity': d.get('popularity'),
        'vote_average': d.get('vote_average'),
        'vote_count': d.get('vote_count'),
        'adult': d.get('adult'),
        'video': d.get('video'),
        'genres': names(d.get('genres')),
        'production_companies': names(d.get('production_companies')),
        'production_countries': names(d.get('production_countries')),
        'spoken_languages': names(d.get('spoken_languages'), 'english_name'),
        'collection_name': coll['name'] if coll else None,
        'director': directors[0] if directors else None,
        'top_cast': '|'.join(c['name'] for c in cast[:3]) or None,
        'cast_size': len(cast),
        'crew_size': len(crew),
        'tagline': d.get('tagline') or None,
        'overview': d.get('overview') or None,
        'imdb_id': d.get('imdb_id'),
        'collected_at': w['collected_at'],
    })

raw_df = pd.DataFrame(rows)
RAW_CSV = f'{RAW_DIR}/tmdb_raw.csv'
raw_df.to_csv(RAW_CSV, index=False, encoding='utf-8')

print('Rows:', raw_df.shape[0], '| Columns:', raw_df.shape[1])
print('Duplicate IDs:', raw_df['id'].duplicated().sum())
raw_df.head()

Rows: 1100 | Columns: 28
Duplicate IDs: 0


,id,title,original_title,release_date,discover_year,status,original_language,runtime,budget,revenue,...,spoken_languages,collection_name,director,top_cast,cast_size,crew_size,tagline,overview,imdb_id,collected_at
0,293660,Deadpool,Deadpool,2016-02-09,2016,Released,en,108,58000000,782837347,...,English,Deadpool Collection,Tim Miller,Ryan Reynolds|Morena Baccarin|Ed Skrein,52,489,Feel the love.,The origin story of former Special Forces oper...,tt1431045,2026-11-21T19:53:17
1,271110,Captain America: Civil War,Captain America: Civil War,2016-04-27,2016,Released,en,147,250000000,1155046416,...,Romanian|English|German|Russian,Captain America Collection,Joe Russo,Chris Evans|Robert Downey Jr.|Scarlett Johansson,104,296,United we stand. Divided we fall.,"Following the events of Age of Ultron, the col...",tt3498820,2026-11-21T19:53:17
2,284052,Doctor Strange,Doctor Strange,2016-10-25,2016,Released,en,115,180000000,676343174,...,English,Doctor Strange Collection,Scott Derrickson,Benedict Cumberbatch|Chiwetel Ejiofor|Rachel M...,44,253,The impossibilities are endless.,"After his career is destroyed, a brilliant but...",tt1211837,2026-11-21T19:53:18
3,297761,Suicide Squad,Suicide Squad,2016-08-03,2016,Released,en,122,175000000,749200054,...,English|Japanese|Spanish,Suicide Squad Collection,David Ayer,Will Smith|Jared Leto|Margot Robbie,106,194,Worst. Heroes. Ever.,"From DC Comics comes the Suicide Squad, an ant...",tt1386697,2026-11-21T19:53:19
4,329865,Arrival,Arrival,2016-11-10,2016,Released,en,116,47000000,203388186,...,Mandarin|English|Russian,None,Denis Villeneuve,Amy Adams|Jeremy Renner|Forest Whitaker,72,487,Why are they here?,Taking place after alien crafts land around th...,tt2543164,2026-11-21T19:53:20


I read the 1100 saved JSON files and combined them into a single table with one movie per row, then saved it as tmdb_raw.csv on Google Drive. I chose attributes from several groups: numbers such as budget, revenue, runtime and vote count, categories such as language, status and genre, dates such as the release date, and true or false values such as adult and video. Fields that hold several values, for example genres and production companies, were joined into one text value separated by a vertical bar so I can split them later during cleaning. I also took the director and the three top billed actors from the credits, and I kept a collected_at timestamp in every row. I did not change any values in this step, so zero budgets and zero revenues are still there and I will report them in the data cleaning section. The output shows 1100 rows and no repeated IDs.

In [ ]:
log = {
    'roll_numbers': ROLL_NUMBERS,
    'start': '2026-11-21T19:51:32',
    'end': '2026-11-21T20:58:43',
    'requests': 1210,
    'note': ('First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). '
             'The discover step was rerun at 20:58 (55 requests) to fix a summary column; '
             'the 1100 saved detail files were reused, not downloaded again.')
}
with open(f'{RAW_DIR}/collection_log.json', 'w') as f:
    json.dump(log, f)
print(log)

{'roll_numbers': ['25L-3003', '25L-0025'], 'start': '2026-11-21T19:51:32', 'end': '2026-11-21T20:58:43', 'requests': 1210, 'note': 'First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). The discover step was rerun at 20:58 (55 requests) to fix a summary column; the 1100 saved detail files were reused, not downloaded again.'}


I corrected the saved collection log file so that it matches what really happened during the collection. When I reran the Discover step to fix a page count in my summary table, the log file was overwritten with the new start time and a request count of only 55. I restored the real values: the collection started at 19:51 and the last rerun ended at 20:58, and the total was 1210 requests, which is 1155 from the first full run plus 55 from the Discover rerun.  The output shows the corrected values. The dataset itself was not changed, because the 1100 detail files had already been saved and were reused.

In [ ]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

with open(f'{RAW_DIR}/collection_log.json') as f:
    log = json.load(f)

fingerprint = sha256_of(RAW_CSV)

print('=== DATA COLLECTION LOG ===')
print('Team roll numbers      :', ', '.join(log['roll_numbers']))
print('Source                 : TMDB API', BASE_URL)
print('Endpoints used         : /discover/movie, /movie/{id}?append_to_response=credits')
print('Collection start       :', log['start'])
print('Collection end         :', log['end'])
print('Number of requests     :', log['requests'], '(55 + 1100 first run, 55 discover rerun)')
print('Note                   :', log['note'])
print('Rows collected         :', len(raw_df))
print('collected_at column    : present | earliest', raw_df['collected_at'].min(),
      '| latest', raw_df['collected_at'].max())
print('Raw file               :', RAW_CSV)
print('SHA-256 fingerprint    :', fingerprint)

with open(f'{RAW_DIR}/data_collection_log.txt', 'w') as f:
    f.write(f"Roll numbers: {log['roll_numbers']}\nStart: {log['start']}\nEnd: {log['end']}\n"
            f"Requests: {log['requests']}\nNote: {log['note']}\nRows: {len(raw_df)}\nSHA-256: {fingerprint}\n")

=== DATA COLLECTION LOG ===
Team roll numbers      : 25L-3003, 25L-0025
Source                 : TMDB API https://api.themoviedb.org/3
Endpoints used         : /discover/movie, /movie/{id}?append_to_response=credits
Collection start       : 2026-11-21T19:51:32
Collection end         : 2026-11-21T20:58:43
Number of requests     : 1210 (55 + 1100 first run, 55 discover rerun)
Note                   : First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). The discover step was rerun at 20:58 (55 requests) to fix a summary column; the 1100 saved detail files were reused, not downloaded again.
Rows collected         : 1100
collected_at column    : present | earliest 2026-11-21T19:53:17 | latest 2026-11-21T20:07:55
Raw file               : /content/drive/MyDrive/AI_Project/data/raw/tmdb_raw.csv
SHA-256 fingerprint    : 45903e104914bc20dc3352cf6914673c628396a9ecb1d7fd110cd92a167b5447


This cell is my data collection log. It records my team's roll numbers, the source and endpoints, the start and end time, the number of requests, the number of rows, the collected_at column and the SHA-256 fingerprint of the raw CSV file. The first full collection ran from 19:51 to about 20:08 and used 1155 requests. The Discover step was rerun once with 55 more requests, and the 1100 detail files I had already saved were reused, so the log records 1210 requests in total. The collected_at values in the rows still show the original collection time. The fingerprint is calculated from the raw CSV file itself, so it works as proof that the file I submit is the one my own collection produced. From now on I will not overwrite tmdb_raw.csv, and all cleaning will be done on copies.

## SECTION **3**: Dataset Construction and Feature Selection



In [6]:
dictionary = pd.DataFrame([
    ('id', 'TMDB movie identifier', 'identifier', 'excluded'),
    ('title', 'Movie title in English', 'text', 'excluded'),
    ('original_title', 'Title in the original language', 'text', 'excluded'),
    ('release_date', 'Date of release (YYYY-MM-DD)', 'date', 'feature (year, month, weekday derived)'),
    ('discover_year', 'Year used in the Discover query', 'numeric', 'excluded (collection artefact)'),
    ('status', 'Release status', 'categorical', 'excluded (near constant)'),
    ('original_language', 'Original language code', 'categorical', 'feature'),
    ('runtime', 'Length in minutes (0 means unknown)', 'numeric', 'feature'),
    ('budget', 'Production budget in USD (0 means unknown)', 'numeric', 'feature'),
    ('revenue', 'Worldwide revenue in USD (0 means unknown)', 'numeric', 'TARGET'),
    ('popularity', 'TMDB popularity score at collection time', 'numeric', 'excluded from model (post-release)'),
    ('vote_average', 'Mean audience rating (0 to 10)', 'numeric', 'excluded from model (post-release)'),
    ('vote_count', 'Number of audience votes', 'numeric', 'excluded from model (post-release)'),
    ('adult', 'Adult content flag', 'boolean', 'excluded (near constant)'),
    ('video', 'Video release flag', 'boolean', 'excluded (near constant)'),
    ('genres', 'Genres, separated by |', 'multi-valued categorical', 'feature (multi-hot)'),
    ('production_companies', 'Companies, separated by |', 'multi-valued categorical', 'feature (count derived)'),
    ('production_countries', 'Countries, separated by |', 'multi-valued categorical', 'feature (count derived)'),
    ('spoken_languages', 'Languages, separated by |', 'multi-valued categorical', 'feature (count derived)'),
    ('collection_name', 'Franchise or collection name', 'categorical', 'feature (is_part_of_collection derived)'),
    ('director', 'First credited director', 'categorical', 'candidate feature (high cardinality)'),
    ('top_cast', 'Three top billed actors, separated by |', 'multi-valued categorical', 'candidate feature'),
    ('cast_size', 'Number of credited cast members', 'numeric', 'candidate feature'),
    ('crew_size', 'Number of credited crew members', 'numeric', 'candidate feature'),
    ('tagline', 'Marketing tagline', 'text', 'excluded (optional text technique)'),
    ('overview', 'Plot summary', 'text', 'excluded (optional text technique)'),
    ('imdb_id', 'IMDb identifier', 'identifier', 'excluded'),
    ('collected_at', 'Time the row was collected', 'date-time', 'excluded'),
], columns=['column', 'meaning', 'kind', 'role'])

dictionary['pandas_dtype'] = dictionary['column'].map(raw_df.dtypes.astype(str))
print('Columns documented:', len(dictionary), '| columns in dataset:', raw_df.shape[1])
dictionary

Columns documented: 28 | columns in dataset: 28


,column,meaning,kind,role,pandas_dtype
0,id,TMDB movie identifier,identifier,excluded,int64
1,title,Movie title in English,text,excluded,object
2,original_title,Title in the original language,text,excluded,object
3,release_date,Date of release (YYYY-MM-DD),date,"feature (year, month, weekday derived)",object
4,discover_year,Year used in the Discover query,numeric,excluded (collection artefact),int64
5,status,Release status,categorical,excluded (near constant),object
6,original_language,Original language code,categorical,feature,object
7,runtime,Length in minutes (0 means unknown),numeric,feature,int64
8,budget,Production budget in USD (0 means unknown),numeric,feature,int64
9,revenue,Worldwide revenue in USD (0 means unknown),numeric,TARGET,int64


I wrote a data dictionary that lists every column with its meaning, its kind (numeric, categorical, boolean, date or text), its pandas data type and the role it will play in the project. I chose revenue as the target because it is a numerical value that can be predicted with regression. I decided that the prediction is made before a movie is released, so audience statistics such as vote average, vote count and popularity cannot be used as inputs, since they only exist after people have watched the film and would leak information about the revenue. I will still use them in the exploratory analysis. Identifiers and titles are excluded so they are not treated as numerical features, and fields that are almost constant, such as adult, video and status, would not help a model. Budget, runtime, release date parts, language, genres, companies, countries, collection membership, the director and the credit counts remain as candidate inputs, and the final set will be decided during feature selection. A limitation of this choice is that a pre-release model will likely be less accurate than one that uses audience votes.


SECTION 4: Data Understanding and Description

In [7]:
print('Rows:', raw_df.shape[0], '| Columns:', raw_df.shape[1])
print('Columns:', list(raw_df.columns))
display(raw_df.head())
display(raw_df.tail())




Rows: 1100 | Columns: 28
Columns: ['id', 'title', 'original_title', 'release_date', 'discover_year', 'status', 'original_language', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'adult', 'video', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'collection_name', 'director', 'top_cast', 'cast_size', 'crew_size', 'tagline', 'overview', 'imdb_id', 'collected_at']


,id,title,original_title,release_date,discover_year,status,original_language,runtime,budget,revenue,...,spoken_languages,collection_name,director,top_cast,cast_size,crew_size,tagline,overview,imdb_id,collected_at
0,293660,Deadpool,Deadpool,2016-02-09,2016,Released,en,108,58000000,782837347,...,English,Deadpool Collection,Tim Miller,Ryan Reynolds|Morena Baccarin|Ed Skrein,52,489,Feel the love.,The origin story of former Special Forces oper...,tt1431045,2026-11-21T19:53:17
1,271110,Captain America: Civil War,Captain America: Civil War,2016-04-27,2016,Released,en,147,250000000,1155046416,...,Romanian|English|German|Russian,Captain America Collection,Joe Russo,Chris Evans|Robert Downey Jr.|Scarlett Johansson,104,296,United we stand. Divided we fall.,"Following the events of Age of Ultron, the col...",tt3498820,2026-11-21T19:53:17
2,284052,Doctor Strange,Doctor Strange,2016-10-25,2016,Released,en,115,180000000,676343174,...,English,Doctor Strange Collection,Scott Derrickson,Benedict Cumberbatch|Chiwetel Ejiofor|Rachel M...,44,253,The impossibilities are endless.,"After his career is destroyed, a brilliant but...",tt1211837,2026-11-21T19:53:18
3,297761,Suicide Squad,Suicide Squad,2016-08-03,2016,Released,en,122,175000000,749200054,...,English|Japanese|Spanish,Suicide Squad Collection,David Ayer,Will Smith|Jared Leto|Margot Robbie,106,194,Worst. Heroes. Ever.,"From DC Comics comes the Suicide Squad, an ant...",tt1386697,2026-11-21T19:53:19
4,329865,Arrival,Arrival,2016-11-10,2016,Released,en,116,47000000,203388186,...,Mandarin|English|Russian,NaN,Denis Villeneuve,Amy Adams|Jeremy Renner|Forest Whitaker,72,487,Why are they here?,Taking place after alien crafts land around th...,tt2543164,2026-11-21T19:53:20


,id,title,original_title,release_date,discover_year,status,original_language,runtime,budget,revenue,...,spoken_languages,collection_name,director,top_cast,cast_size,crew_size,tagline,overview,imdb_id,collected_at
1095,1248832,Digger,Digger,2026-09-28,2026,Released,en,129,180000000,50850116,...,English|Spanish|French|Mandarin,NaN,Alejandro G. Iñárritu,Tom Cruise|Riz Ahmed|John Goodman,89,121,A man. A plan. A meltdown.,The most powerful man in the world embarks on ...,tt31450459,2026-11-21T20:07:52
1096,949838,72 HOURS,72 HOURS,2026-07-20,2026,Released,en,105,0,0,...,English,NaN,Tim Story,Kevin Hart|Marcello Hernández|Mason Gooding,52,29,The night is young. He isn't.,"To save his career, a 40-year-old ad exec join...",tt36304003,2026-11-21T20:07:53
1097,1560520,Batman: Knightfall Part 1: Knightfall,Batman: Knightfall Part 1: Knightfall,2026-08-25,2026,Released,en,79,0,0,...,English|Spanish,Batman: Knightfall Collection,Jeff Wamester,Anson Mount|Michael Mando|Pablo Schreiber,22,143,NaN,"Arkham Asylum has been destroyed, and all its ...",tt32333324,2026-11-21T20:07:53
1098,1115544,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,2026-03-14,2026,Released,en,107,0,0,...,English,NaN,BenDavid Grabinski,James Marsden|Vince Vaughn|Eiza González,35,46,"A time-traveling, double-crossing, ass-kicking...",Two gangsters and the woman they love try to s...,tt27552099,2026-11-21T20:07:54
1099,1116201,Iron Lung,Iron Lung,2026-01-30,2026,Released,en,125,3000000,51226173,...,English,NaN,Mark Fischbach,Mark Fischbach|Caroline Kaplan|Troy Baker,23,159,This is not an expedition. It's an execution.,In a post-apocalyptic future where an event kn...,tt27564844,2026-11-21T20:07:55


I displayed the first and last five records and checked the size and column names of the raw dataset. I did this to get a first feel for the data and to confirm that the table was built correctly.The first rows come from 2016 and the last rows from 2026, because the data was collected year by year. The columns match the data dictionary I wrote earlier.

In [8]:
raw_df.info()
print()
print('Numeric   :', raw_df.select_dtypes('number').columns.tolist())
print('Boolean   :', raw_df.select_dtypes('bool').columns.tolist())
print('Text/categorical:', raw_df.select_dtypes('object').columns.tolist())
print('\nMemory usage (MB):', round(raw_df.memory_usage(deep=True).sum() / 1e6, 2))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    1100 non-null   int64  
 1   title                 1100 non-null   object 
 2   original_title        1100 non-null   object 
 3   release_date          1100 non-null   object 
 4   discover_year         1100 non-null   int64  
 5   status                1100 non-null   object 
 6   original_language     1100 non-null   object 
 7   runtime               1100 non-null   int64  
 8   budget                1100 non-null   int64  
 9   revenue               1100 non-null   int64  
 10  popularity            1100 non-null   float64
 11  vote_average          1100 non-null   float64
 12  vote_count            1100 non-null   int64  
 13  adult                 1100 non-null   bool   
 14  video                 1100 non-null   bool   
 15  genres               

I used info to check the data types and the number of non-null values in each column, and I grouped the columns into numeric, boolean and text types. I also measured how much memory the dataset uses. This matters because the date columns, release_date and collected_at, were read as plain text and will have to be converted to dates later. The multi-valued fields, such as genres, are text and will need splitting before encoding. The dataset is small, using about [X] MB, so memory is not a concern.

In [9]:
display(raw_df.describe().T)
display(raw_df.describe(include='object').T)

,count,mean,std,min,25%,50%,75%,max
id,1100.0,6.478979e+05,3.185277e+05,14564.0000,4.018530e+05,5.652270e+05,8.412598e+05,1.698863e+06
discover_year,1100.0,2.021000e+03,3.163716e+00,2016.0000,2.018000e+03,2.021000e+03,2.024000e+03,2.026000e+03
runtime,1100.0,1.151191e+02,2.030853e+01,12.0000,1.020000e+02,1.120000e+02,1.260000e+02,2.420000e+02
budget,1100.0,6.165258e+07,7.476431e+07,0.0000,9.000000e+06,3.500000e+07,9.000000e+07,6.588000e+08
revenue,1100.0,1.882582e+08,3.210708e+08,0.0000,6.151455e+06,6.083273e+07,2.139270e+08,2.942661e+09
popularity,1100.0,1.214923e+01,2.710758e+01,0.0541,5.004025e+00,7.319800e+00,1.153540e+01,4.817218e+02
vote_average,1100.0,6.943792e+00,6.741157e-01,3.9760,6.484500e+00,6.937500e+00,7.393250e+00,9.093000e+00
vote_count,1100.0,4.633278e+03,4.394863e+03,490.0000,1.863500e+03,3.180500e+03,5.585000e+03,3.366000e+04
cast_size,1100.0,4.847091e+01,3.406892e+01,0.0000,2.900000e+01,4.100000e+01,6.000000e+01,3.940000e+02
crew_size,1100.0,1.470682e+02,1.077676e+02,1.0000,7.475000e+01,1.220000e+02,1.840000e+02,8.300000e+02


,count,unique,top,freq
title,1100,1096,Deep Water,2
original_title,1100,1096,Deep Water,2
release_date,1100,870,2021-11-24,5
status,1100,1,Released,1100
original_language,1100,16,en,1021
genres,1100,493,Drama,33
production_companies,1099,992,Pixar,14
production_countries,1098,176,United States of America,651
spoken_languages,1095,242,English,640
collection_name,392,256,The Fast and the Furious Collection,4


I generated descriptive statistics for the numeric columns and for the text columns. For the numeric ones I looked at the mean, spread, minimum and maximum, and for the text ones the number of unique values and the most common value. I noticed that the minimum of budget, revenue and runtime is 0, which in TMDB means the value is unknown and not a real zero, and the large gap between the mean and the maximum in budget and revenue suggests strong skewness. This will guide my cleaning and my treatment of skewness later.

In [10]:
for col in ['original_language', 'status', 'adult', 'video', 'collection_name', 'director']:
    print(f'--- {col}: {raw_df[col].nunique()} unique values')
    print(raw_df[col].value_counts().head(8), '\n')

genre_counts = raw_df['genres'].dropna().str.split('|').explode().value_counts()
print('--- genres (multi-valued):', len(genre_counts), 'unique genres')
print(genre_counts)

--- original_language: 16 unique values
original_language
en    1021
ja      17
fr      14
es      13
ko       9
it       6
no       4
zh       3
Name: count, dtype: int64 

--- status: 1 unique values
status
Released    1100
Name: count, dtype: int64 

--- adult: 1 unique values
adult
False    1100
Name: count, dtype: int64 

--- video: 1 unique values
video
False    1100
Name: count, dtype: int64 

--- collection_name: 256 unique values
collection_name
The Fast and the Furious Collection            4
Spider-Man (MCU) Collection                    4
Deadpool Collection                            3
Fantastic Beasts Collection                    3
Hercule Poirot (Kenneth Branagh) Collection    3
John Wick Collection                           3
The Conjuring Collection                       3
The Purge Collection                           3
Name: count, dtype: int64 

--- director: 679 unique values
director
Guy Ritchie           9
Antoine Fuqua         6
Ridley Scott          6
Steven S

I counted the unique values and the most frequent values of the categorical columns, and I split the genres field to count each genre separately, since a movie can have several genres. I wanted to know which columns are suitable for one-hot encoding and which have too many categories. Language has a few dominant values, status, adult and video are almost constant, and director and collection name have many unique values. The genre counts show which genres are common and which are rare. This tells me that director needs a different treatment than simple one-hot encoding.

In [11]:
missing = raw_df.isnull().sum().to_frame('missing_NaN')
missing['percent'] = (missing['missing_NaN'] / len(raw_df) * 100).round(2)
display(missing[missing['missing_NaN'] > 0])

zero_cols = ['budget', 'revenue', 'runtime']
zeros = pd.DataFrame({'zeros': [(raw_df[c] == 0).sum() for c in zero_cols]}, index=zero_cols)
zeros['percent'] = (zeros['zeros'] / len(raw_df) * 100).round(2)
display(zeros)

print('Duplicate rows:', raw_df.duplicated().sum())
print('Duplicate movie IDs:', raw_df['id'].duplicated().sum())

,missing_NaN,percent
production_companies,1,0.09
production_countries,2,0.18
spoken_languages,5,0.45
collection_name,708,64.36
top_cast,2,0.18
tagline,43,3.91
imdb_id,1,0.09


,zeros,percent
budget,160,14.55
revenue,201,18.27
runtime,0,0.00


Duplicate rows: 0
Duplicate movie IDs: 0


I checked every column for missing values and counted the zeros in budget, revenue and runtime, because TMDB stores unknown values of these fields as 0 and not as an empty cell. I also checked for duplicate rows and repeated movie IDs. The output shows which columns have real missing values, such as tagline or collection name, and how many movies have a zero budget, revenue or runtime. There are no duplicate rows or repeated IDs, which agrees with how I collected the data. I will treat the zeros as missing values in the cleaning section.